# Прогнозирование оттока клиентов телеком-компании

В проекте сравниваются логистическая регрессия, случайный лес и градиентный бустинг. Основная метрика — F1-score. Балансировка классов и порог классификации выбираются только по обучающей части с помощью 5-кратной стратифицированной кросс-валидации.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
USE_CACHE = False
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

## 1. Загрузка и очистка данных

In [2]:
data_path = Path("iranian_churn.csv")
if not data_path.exists():
    data_path = Path("/mnt/data/iranian_churn.csv")
if not data_path.exists():
    raise FileNotFoundError("Файл iranian_churn.csv не найден")

raw_df = pd.read_csv(data_path)
duplicate_count = raw_df.duplicated().sum()
df = raw_df.drop_duplicates().reset_index(drop=True)

print(f"Исходный размер: {raw_df.shape[0]} строк, {raw_df.shape[1]} столбцов")
print(f"Полных дубликатов удалено: {duplicate_count}")
print(f"Размер после очистки: {df.shape[0]} строк, {df.shape[1]} столбцов")
display(df.head())

Исходный размер: 3150 строк, 16 столбцов
Полных дубликатов удалено: 300
Размер после очистки: 2850 строк, 16 столбцов


,Call Failure,Complains,Subscription Length,Charge Amount,Seconds of Use,Frequency of use,Frequency of SMS,Distinct Called Numbers,Age Group,Tariff Plan,Status,Age,Customer Value,FN,FP,Churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.6400,177.8760,69.7640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.0350,41.4315,60.0000,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.5200,1382.8680,203.6520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.0200,216.0180,74.0020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.8050,131.2245,64.5805,0


In [3]:
overview = pd.DataFrame({
    "Тип": df.dtypes.astype(str),
    "Пропуски": df.isna().sum(),
    "Уникальные значения": df.nunique(),
})
display(overview)
print("Доля класса Churn=1:", round(df["Churn"].mean(), 4))

,Тип,Пропуски,Уникальные значения
Call Failure,int64,0,37
Complains,int64,0,2
Subscription Length,int64,0,45
Charge Amount,int64,0,11
Seconds of Use,int64,0,1756
Frequency of use,int64,0,242
Frequency of SMS,int64,0,405
Distinct Called Numbers,int64,0,92
Age Group,int64,0,5
Tariff Plan,int64,0,2


Доля класса Churn=1: 0.1565


In [4]:
class_counts = df["Churn"].value_counts().sort_index()
ax = class_counts.plot(kind="bar", figsize=(7, 4))
ax.set_title("Распределение целевого класса")
ax.set_xlabel("Churn")
ax.set_ylabel("Количество клиентов")
ax.set_xticklabels(["Остался", "Ушёл"], rotation=0)
plt.tight_layout()
plt.show()

/tmp/ipykernel_13837/398996318.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Подготовка признаков

`FN` и `FP` исключаются как технические производные столбцы. Полные дубликаты удаляются до разбиения, чтобы одинаковые записи не оказались одновременно в обучении и тесте.

In [5]:
target = "Churn"
feature_columns = [column for column in df.columns if column not in [target, "FN", "FP"]]
X = df[feature_columns]
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Количество признаков:", len(feature_columns))
print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)
display(y_train.value_counts(normalize=True).sort_index().rename("Доля").to_frame())

Количество признаков: 13
Обучающая выборка: (2280, 13)
Тестовая выборка: (570, 13)


,Доля
Churn,
0,0.8434
1,0.1566


## 3. Схема эксперимента

Вспомогательный файл `model_worker.py` изолированно запускает каждую модель. Это не меняет алгоритм оценки, но делает выполнение устойчивым: каждая конфигурация получает собственный процесс Python.

In [6]:
worker_path = Path("model_worker.py")
if not worker_path.exists():
    worker_path = Path("/mnt/data/model_worker.py")
if not worker_path.exists():
    raise FileNotFoundError("Файл model_worker.py не найден")

model_keys = ["logistic", "random_forest", "gradient_boosting"]


def run_worker(model_key, mode, balanced, output_path, threshold=0.5):
    output_path = Path(output_path)
    if not USE_CACHE or not output_path.exists():
        environment = os.environ.copy()
        environment.update({
            "OPENBLAS_NUM_THREADS": "1",
            "OMP_NUM_THREADS": "1",
            "MKL_NUM_THREADS": "1",
        })
        command = [
            sys.executable,
            str(worker_path),
            "--model", model_key,
            "--mode", mode,
            "--balanced", str(int(balanced)),
            "--threshold", str(threshold),
            "--data", str(data_path),
            "--output", str(output_path),
        ]
        subprocess.run(command, check=True, env=environment)
    return json.loads(output_path.read_text(encoding="utf-8"))

## 4. A/B-тест балансировки классов

In [7]:
cv_results_raw = []
for model_key in model_keys:
    for balanced in [False, True]:
        cv_results_raw.append(run_worker(
            model_key=model_key,
            mode="cv",
            balanced=balanced,
            output_path=f"cv_{model_key}_{int(balanced)}.json",
        ))
print("Кросс-валидация завершена для 6 конфигураций")

Кросс-валидация завершена для 6 конфигураций


In [8]:
ab_rows = []
fold_rows = []
for result in cv_results_raw:
    balance_label = "С балансировкой" if result["balanced"] else "Без балансировки"
    ab_rows.append({
        "Модель": result["model_name"],
        "Балансировка": balance_label,
        "Порог": result["threshold"],
        "CV F1 при пороге 0.5": result["default_metrics"]["f1"],
        "CV F1 после подбора порога": result["tuned_metrics"]["f1"],
        "CV Precision": result["tuned_metrics"]["precision"],
        "CV Recall": result["tuned_metrics"]["recall"],
        "CV ROC-AUC": result["tuned_metrics"]["roc_auc"],
        "CV Average Precision": result["tuned_metrics"]["average_precision"],
    })
    for row in result["folds"]:
        fold_rows.append({
            "Модель": result["model_name"],
            "Балансировка": balance_label,
            **row,
        })

ab_results = pd.DataFrame(ab_rows).sort_values(
    ["Модель", "CV F1 после подбора порога"],
    ascending=[True, False],
).reset_index(drop=True)
cross_validation_results = pd.DataFrame(fold_rows)
display(ab_results)

,Модель,Балансировка,Порог,CV F1 при пороге 0.5,CV F1 после подбора порога,CV Precision,CV Recall,CV ROC-AUC,CV Average Precision
0,Градиентный бустинг,Без балансировки,0.3750,0.8467,0.8607,0.8560,0.8655,0.9815,0.9277
1,Градиентный бустинг,С балансировкой,0.6600,0.8379,0.8472,0.8402,0.8543,0.9808,0.9189
2,Логистическая регрессия,Без балансировки,0.2500,0.5785,0.6814,0.5825,0.8207,0.9330,0.7505
3,Логистическая регрессия,С балансировкой,0.6600,0.6353,0.6810,0.5938,0.7983,0.9335,0.7469
4,Случайный лес,Без балансировки,0.4050,0.8211,0.8628,0.8714,0.8543,0.9824,0.9216
5,Случайный лес,С балансировкой,0.5300,0.8389,0.8420,0.8196,0.8655,0.9792,0.8991


In [9]:
plot_data = ab_results.pivot(
    index="Модель",
    columns="Балансировка",
    values="CV F1 после подбора порога",
)
ax = plot_data.plot(kind="bar", figsize=(10, 5))
ax.set_title("A/B-тест балансировки классов")
ax.set_ylabel("OOF F1-score")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

/tmp/ipykernel_13837/745212208.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
selected_configurations = {}
for result in cv_results_raw:
    current = selected_configurations.get(result["model_key"])
    current_f1 = -1 if current is None else current["tuned_metrics"]["f1"]
    if result["tuned_metrics"]["f1"] > current_f1:
        selected_configurations[result["model_key"]] = result

selection_table = pd.DataFrame([
    {
        "Модель": result["model_name"],
        "Выбранный режим": "С балансировкой" if result["balanced"] else "Без балансировки",
        "Порог": result["threshold"],
        "CV F1": result["tuned_metrics"]["f1"],
        "CV ROC-AUC": result["tuned_metrics"]["roc_auc"],
    }
    for result in selected_configurations.values()
]).sort_values("CV F1", ascending=False)
display(selection_table)

,Модель,Выбранный режим,Порог,CV F1,CV ROC-AUC
1,Случайный лес,Без балансировки,0.4050,0.8628,0.9824
2,Градиентный бустинг,Без балансировки,0.3750,0.8607,0.9815
0,Логистическая регрессия,Без балансировки,0.2500,0.6814,0.9330


## 5. Кросс-валидация по всем признакам

In [11]:
cross_validation_summary = (
    cross_validation_results
    .groupby(["Модель", "Балансировка"])[
        ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
    ]
    .agg(["mean", "std"])
    .round(4)
)
display(cross_validation_summary)

accuracy        precision         \
                                             mean    std      mean    std   
Модель                  Балансировка                                        
Градиентный бустинг     Без балансировки   0.9539 0.0113    0.8842 0.0385   
                        С балансировкой    0.9452 0.0107    0.7806 0.0332   
Логистическая регрессия Без балансировки   0.8952 0.0088    0.7818 0.0461   
                        С балансировкой    0.8434 0.0108    0.5002 0.0196   
Случайный лес           Без балансировки   0.9478 0.0117    0.8866 0.0470   
                        С балансировкой    0.9469 0.0126    0.7993 0.0331   

                                         recall            f1        roc_auc  \
                                           mean    std   mean    std    mean   
Модель                  Балансировка                                           
Градиентный бустинг     Без балансировки 0.8121 0.0491 0.8462 0.0404  0.9818   
                        С балансировкой  0.9046 0.0415 0.8377 0.0321  0.9811   
Логистическая регрессия Без балансировки 0.4593 0.0445 0.5777 0.0418  0.9330   
                        С балансировкой  0.8710 0.0459 0.6352 0.0243  0.9337   
Случайный лес           Без балансировки 0.7646 0.0445 0.8208 0.0421  0.9825   
                        С балансировкой  0.8822 0.0618 0.8382 0.0414  0.9789   

                                                average_precision         
                                            std              mean    std  
Модель                  Балансировка                                      
Градиентный бустинг     Без балансировки 0.0069            0.9289 0.0272  
                        С балансировкой  0.0063            0.9209 0.0285  
Логистическая регрессия Без балансировки 0.0084            0.7508 0.0241  
                        С балансировкой  0.0071            0.7459 0.0246  
Случайный лес           Без балансировки 0.0083            0.9230 0.0365  
                        С балансировкой  0.0099            0.8982 0.0491

## 6. Итоговая оценка на независимом тесте

In [12]:
final_results_raw = []
for model_key, configuration in selected_configurations.items():
    final_results_raw.append(run_worker(
        model_key=model_key,
        mode="final",
        balanced=configuration["balanced"],
        threshold=configuration["threshold"],
        output_path=f"final_{model_key}.json",
    ))
print("Итоговые модели обучены")

Итоговые модели обучены


In [13]:
test_rows = []
for result in final_results_raw:
    configuration = selected_configurations[result["model_key"]]
    metrics = result["metrics"]
    test_rows.append({
        "Модель": result["model_name"],
        "Балансировка": "С балансировкой" if result["balanced"] else "Без балансировки",
        "Порог": result["threshold"],
        "CV F1": configuration["tuned_metrics"]["f1"],
        "Test Accuracy": metrics["accuracy"],
        "Test Precision": metrics["precision"],
        "Test Recall": metrics["recall"],
        "Test F1": metrics["f1"],
        "Test ROC-AUC": metrics["roc_auc"],
        "Test Average Precision": metrics["average_precision"],
    })

model_results = pd.DataFrame(test_rows).sort_values("Test F1", ascending=False).reset_index(drop=True)
display(model_results)

,Модель,Балансировка,Порог,CV F1,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,Test Average Precision
0,Случайный лес,Без балансировки,0.4050,0.8628,0.9667,0.9070,0.8764,0.8914,0.9847,0.9340
1,Градиентный бустинг,Без балансировки,0.3750,0.8607,0.9596,0.8438,0.9101,0.8757,0.9872,0.9481
2,Логистическая регрессия,Без балансировки,0.2500,0.6814,0.8702,0.5591,0.7978,0.6574,0.9276,0.7614


In [14]:
ax = model_results.set_index("Модель")["Test F1"].plot(kind="bar", figsize=(9, 5))
ax.set_title("Итоговый F1-score на тестовой выборке")
ax.set_ylabel("F1-score")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

/tmp/ipykernel_13837/519266065.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
plt.figure(figsize=(8, 6))
for result in final_results_raw:
    probabilities = np.array(result["probabilities"])
    y_true = np.array(result["y_test"])
    false_positive_rate, true_positive_rate, _ = roc_curve(y_true, probabilities)
    auc_value = roc_auc_score(y_true, probabilities)
    plt.plot(false_positive_rate, true_positive_rate, label=f"{result['model_name']}: {auc_value:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.title("ROC-кривые на тестовой выборке")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()

/tmp/ipykernel_13837/1064191071.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Какая модель лучше?

In [16]:
best_model_row = model_results.iloc[0]
best_result = next(
    result for result in final_results_raw
    if result["model_name"] == best_model_row["Модель"]
)

answer = (
    f"**Лучшая модель — {best_model_row['Модель']}.** "
    f"На независимой тестовой выборке она получила F1 = {best_model_row['Test F1']:.3f}, "
    f"Precision = {best_model_row['Test Precision']:.3f}, "
    f"Recall = {best_model_row['Test Recall']:.3f} и "
    f"ROC-AUC = {best_model_row['Test ROC-AUC']:.3f}."
)
display(Markdown(answer))

**Лучшая модель — Случайный лес.** На независимой тестовой выборке она получила F1 = 0.891, Precision = 0.907, Recall = 0.876 и ROC-AUC = 0.985.

In [17]:
matrix = np.array(best_result["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5, 5))
image = ax.imshow(matrix)
for row in range(matrix.shape[0]):
    for column in range(matrix.shape[1]):
        ax.text(column, row, matrix[row, column], ha="center", va="center")
ax.set_title(f"Матрица ошибок: {best_result['model_name']}")
ax.set_xlabel("Предсказанный класс")
ax.set_ylabel("Истинный класс")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
plt.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

/tmp/ipykernel_13837/3217825532.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Анализ важности признаков

In [18]:
feature_importance = pd.DataFrame({
    "Признак": best_result["features"],
    "Важность": best_result["importance"],
    "Метод": best_result["importance_method"],
}).sort_values("Важность", ascending=False).reset_index(drop=True)
display(feature_importance)

,Признак,Важность,Метод
0,Complains,0.2294,Встроенная важность
1,Status,0.1279,Встроенная важность
2,Seconds of Use,0.1152,Встроенная важность
3,Frequency of use,0.1038,Встроенная важность
4,Subscription Length,0.1004,Встроенная важность
5,Customer Value,0.0782,Встроенная важность
6,Distinct Called Numbers,0.0703,Встроенная важность
7,Call Failure,0.0613,Встроенная важность
8,Frequency of SMS,0.0464,Встроенная важность
9,Age,0.0275,Встроенная важность


In [19]:
top_features = feature_importance.head(10).sort_values("Важность")
ax = top_features.plot(
    x="Признак",
    y="Важность",
    kind="barh",
    figsize=(9, 6),
    legend=False,
)
ax.set_title(f"Важность признаков: {best_result['model_name']}")
ax.set_xlabel("Важность")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

/tmp/ipykernel_13837/3221620111.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Выводы и рекомендации

In [20]:
gradient_f1 = model_results.loc[
    model_results["Модель"] == "Градиентный бустинг",
    "Test F1",
].iloc[0]
top_names = ", ".join(feature_importance.head(5)["Признак"].tolist())

balance_lines = []
for model_key, result in selected_configurations.items():
    state = "дала лучший CV F1" if result["balanced"] else "не повысила лучший CV F1"
    balance_lines.append(f"- Для модели «{result['model_name']}» балансировка {state}.")

conclusions = f"""
### Основные выводы

- Требование F1 не ниже 0,70 выполнено: лучшая модель получила **{best_model_row['Test F1']:.3f}**.
- Лучшая модель — **{best_model_row['Модель']}**.
- Градиентный бустинг получил Test F1 = **{gradient_f1:.3f}**.
- Самые значимые признаки: **{top_names}**.
- Балансировка повышает Recall при фиксированном пороге, но не обязана повышать максимальный F1.

### A/B-тест балансировки

{chr(10).join(balance_lines)}

### Рекомендации

1. Использовать **{best_model_row['Модель']}** как основную модель оценки риска оттока.
2. Применять градиентный бустинг как сильную альтернативу с более высоким Recall.
3. Направлять удерживающие предложения клиентам с высокой прогнозной вероятностью оттока.
4. Контролировать признаки: {top_names}.
5. Переобучать модели на новых периодах и отслеживать F1 и ROC-AUC.
"""
display(Markdown(conclusions))


### Основные выводы

- Требование F1 не ниже 0,70 выполнено: лучшая модель получила **0.891**.
- Лучшая модель — **Случайный лес**.
- Градиентный бустинг получил Test F1 = **0.876**.
- Самые значимые признаки: **Complains, Status, Seconds of Use, Frequency of use, Subscription  Length**.
- Балансировка повышает Recall при фиксированном пороге, но не обязана повышать максимальный F1.

### A/B-тест балансировки

- Для модели «Логистическая регрессия» балансировка не повысила лучший CV F1.
- Для модели «Случайный лес» балансировка не повысила лучший CV F1.
- Для модели «Градиентный бустинг» балансировка не повысила лучший CV F1.

### Рекомендации

1. Использовать **Случайный лес** как основную модель оценки риска оттока.
2. Применять градиентный бустинг как сильную альтернативу с более высоким Recall.
3. Направлять удерживающие предложения клиентам с высокой прогнозной вероятностью оттока.
4. Контролировать признаки: Complains, Status, Seconds of Use, Frequency of use, Subscription  Length.
5. Переобучать модели на новых периодах и отслеживать F1 и ROC-AUC.


## 10. Сохранение результатов

In [21]:
ab_results.to_csv("class_balance_ab_test.csv", index=False)
cross_validation_results.to_csv("cross_validation_results.csv", index=False)
model_results.to_csv("model_results.csv", index=False)
feature_importance.to_csv("feature_importance.csv", index=False)

requirements = """numpy>=1.24
pandas>=2.0
matplotlib>=3.7
scikit-learn>=1.3
jupyter>=1.0
"""
Path("requirements.txt").write_text(requirements, encoding="utf-8")

result_lines = [
    "| Модель | Балансировка | Порог | CV F1 | Test F1 | Precision | Recall | ROC-AUC |",
    "|---|---|---:|---:|---:|---:|---:|---:|",
]
for _, row in model_results.iterrows():
    result_lines.append(
        f"| {row['Модель']} | {row['Балансировка']} | {row['Порог']:.3f} | "
        f"{row['CV F1']:.3f} | {row['Test F1']:.3f} | {row['Test Precision']:.3f} | "
        f"{row['Test Recall']:.3f} | {row['Test ROC-AUC']:.3f} |"
    )

readme = f"""# Прогнозирование оттока клиентов

Проект сравнивает логистическую регрессию, случайный лес и градиентный бустинг на Iranian Churn Dataset.

## Данные

- исходно 3150 наблюдений;
- удалено {duplicate_count} полных дубликатов;
- после очистки осталось {len(df)} наблюдений;
- используются 13 исходных признаков;
- `FN` и `FP` исключены.

## Методика

- стратифицированное разделение 80/20;
- 5-кратная кросс-валидация;
- A/B-тест балансировки;
- подбор порога по OOF-прогнозам;
- независимая тестовая выборка;
- анализ важности признаков.

## Результаты

{chr(10).join(result_lines)}

**Лучшая модель: {best_model_row['Модель']}.** Test F1 = **{best_model_row['Test F1']:.3f}**.

## Файлы

- `Код_для_практики_iranian_churn_готово.ipynb` — ноутбук;
- `model_worker.py` — запуск моделей;
- `iranian_churn.csv` — датасет;
- `model_results.csv` — итоговые метрики;
- `class_balance_ab_test.csv` — A/B-тест;
- `cross_validation_results.csv` — результаты по фолдам;
- `feature_importance.csv` — важность признаков;
- `requirements.txt` — зависимости.
"""
Path("README.md").write_text(readme, encoding="utf-8")
print("Файлы проекта сохранены")

Файлы проекта сохранены
